# 06. Модель двух пятен TCE

Это подход из ML-006. Концентрация складывается из пятна около источника, движущегося пятна и фона. Параметры заново оцениваются по замерам текущего набора.

В этой модели оба пятна описывают поле приближённо. Откачка учитывается при смешивании воды на водозаборе.

## 1. Чтение исходных таблиц

In [1]:
from scipy.optimize import least_squares
from threadpoolctl import threadpool_limits

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

here = Path.cwd()
ROOT = next(folder for folder in (here, *here.parents) if (folder / "data" / "iterim").is_dir())
DATA = ROOT / "data" / "iterim"
MASS_DATE = pd.Timestamp("2026-01-01")
CELL_SIZE = 50  # м
THRESHOLD = 5.0  # мкг/л
LEVEL = 0.90


def read(name):
    return pd.read_parquet(DATA / f"{name}.parquet")


wells = read("wells")
samples = read("concentration").merge(
    wells[["well_id", "x", "y", "z"]], on="well_id", how="left", validate="many_to_one"
)
queries = read("queries_test")
sources = read("sources").sort_values("source_id")
pumping = read("pumping")
layers = read("cores")[["z_top", "z_bottom", "porosity"]].drop_duplicates()
raster = tifffile.imread(ROOT / "data" / "raw" / "permeability.tif")

START = sources["start_date"].min()
OUTPUT = ROOT / "submission_plume"

## 2. Входные данные для модели пятен

Формула работает с x, y, z и числом дней с даты первого источника. При прогнозе на сетке и водозаборе используются те же координаты и отсчёт времени.

In [2]:
def month_start(dates):
    return dates.dt.to_period("M").dt.to_timestamp()


def build_features(points):
    days = (points["t"] - START).dt.days
    return points[["x", "y", "z"]].astype(float).assign(days=days)


X = build_features(samples)
y = samples["concentration"]
display(X.head())

,x,y,z,days
0,970.08,1422.65,-35.83,1223
1,1491.46,1751.22,-46.12,713
2,823.59,1326.11,-16.35,1379
3,223.30,2572.96,-9.08,2289
4,345.08,1762.65,-6.27,2237


## 3. Подбор параметров двух пятен

Направление от основного источника к водозабору используется как начальное приближение. SciPy подбирает направление, скорость, ширину, глубину и амплитуду по обучающим пробам. Сравниваем модель и замеры на шкале log1p, чтобы высокие пики меньше влияли на подбор, и считаем RMSE в мкг/л.

In [3]:
def plume_concentration(parameters, points):
    """Сумма пятна у источника, движущегося пятна и фона."""
    (
        x0,
        y0,
        depth_halo,
        depth_plume,
        log_halo,
        log_plume,
        halo_xy,
        halo_z,
        length,
        width,
        plume_z,
        speed,
        angle,
        length_growth,
        width_growth,
        background,
    ) = parameters
    x, y, z, days = points.T
    dx = x - x0
    dy = y - y0
    along = dx * np.cos(angle) + dy * np.sin(angle) - speed * days
    across = -dx * np.sin(angle) + dy * np.cos(angle)
    halo_radius_sq = (dx**2 + dy**2) / halo_xy**2 + ((z - depth_halo) / halo_z) ** 2
    plume_radius_sq = (
        (along / (length + length_growth * days)) ** 2
        + (across / (width + width_growth * days)) ** 2
        + ((z - depth_plume) / plume_z) ** 2
    )
    halo = np.exp(log_halo - 0.5 * halo_radius_sq)
    plume = np.exp(log_plume - 0.5 * plume_radius_sq)
    return background + halo + plume


def fit_model(features, target):
    # Начальная точка оптимизации берётся из геометрии этого набора.
    main_source = sources.loc[sources["estimated_release_rate_g_day"].idxmax()]
    x0, y0 = float(main_source.x), float(main_source.y)
    source_depth = (main_source.z_top + main_source.z_bottom) / 2
    intake = wells.loc[wells["role"].eq("intake"), ["x", "y"]].mean()
    angle = np.arctan2(intake.y - y0, intake.x - x0)
    amplitude = max(float(target.quantile(0.98)), 1.0)

    # Порядок параметров совпадает с распаковкой в plume_concentration.
    start = [
        x0, y0, source_depth, -32, np.log(amplitude), np.log(amplitude / 2),
        250, 15, 350, 180, 22, 0.8, angle, 0.13, 0.04, 0.03,
    ]
    lower = [
        x0 - 600, y0 - 600, -76, -76, -5, -5,
        50, 3, 60, 40, 5, 0.05, angle - 1, 0, 0, 0,
    ]
    upper = [
        x0 + 600, y0 + 600, -1, -1, 10, 10,
        1300, 90, 1600, 1000, 90, 3, angle + 1, 0.7, 0.5, 1,
    ]

    points = features.to_numpy(dtype=float)
    measured = np.log1p(target.to_numpy(dtype=float))

    def residuals(parameters):
        return np.log1p(plume_concentration(parameters, points)) - measured

    with threadpool_limits(limits=1, user_api="blas"):
        result = least_squares(
            residuals,
            start,
            bounds=(lower, upper),
            x_scale="jac",
            loss="soft_l1",
            f_scale=0.3,
            max_nfev=500,
        )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x


def predict(model, features):
    return plume_concentration(model, features.to_numpy(dtype=float))

## 4. Калибровка интервалов и проверка по времени

Учим модель на данных до 2024 года, подбираем ширину интервала по ошибкам 2024 года и проверяем оба результата на 2025 годе. Для итоговой модели сдвигаем обучение на год вперёд и калибруем по 2025 году. Из-за смещения по времени фактическое покрытие может отличаться от 90%.

In [4]:
def calibrate_interval(model, features, target):
    center = predict(model, features)
    errors = np.abs(target.to_numpy(dtype=float) - center)
    scores = errors / np.sqrt(1 + center)
    rank = min(len(scores), int(np.ceil((len(scores) + 1) * LEVEL)))
    return float(np.partition(scores, rank - 1)[rank - 1])


def predict_interval(model, features, factor):
    center = predict(model, features)
    radius = factor * np.sqrt(1 + center)
    return center, np.maximum(center - radius, 0.0), center + radius

In [5]:
dates = samples["t"]
before_2024 = dates.lt("2024-01-01")
year_2024 = dates.dt.year.eq(2024)
year_2025 = dates.dt.year.eq(2025)
before_2025 = dates.lt("2025-01-01")
assert before_2024.any() and year_2024.any() and year_2025.any()

check_model = fit_model(X.loc[before_2024], y.loc[before_2024])
check_factor = calibrate_interval(check_model, X.loc[year_2024], y.loc[year_2024])
center, low, high = predict_interval(check_model, X.loc[year_2025], check_factor)
actual = y.loc[year_2025].to_numpy(dtype=float)

rmse = np.sqrt(np.mean((actual - center) ** 2))
coverage = np.mean((low <= actual) & (actual <= high))
print(f"Проверка на 2025 годе: RMSE {rmse:.3f} мкг/л, покрытие {coverage:.1%}")

# Модель для выдачи прогнозов использует 2024 год в обучении и 2025 в калибровке.
model = fit_model(X.loc[before_2025], y.loc[before_2025])
interval_factor = calibrate_interval(model, X.loc[year_2025], y.loc[year_2025])

Проверка на 2025 годе: RMSE 6.835 мкг/л, покрытие 88.6%


## 5. Прогноз для queries_test

Строим признаки той же функцией, что и для замеров. Сохраняем по одной строке на запрос с исходным query_id.

In [6]:
X_query = build_features(queries)
center, low, high = predict_interval(model, X_query, interval_factor)
predictions = pd.DataFrame({
    "query_id": queries["query_id"],
    "prediction": center,
    "lower_90": low,
    "upper_90": high,
})
assert len(predictions) == len(queries) and predictions["query_id"].is_unique
display(predictions.head())

,query_id,prediction,lower_90,upper_90
0,Q-0341,7.286792,0.000000,14.866264
1,Q-0488,0.798138,0.000000,4.328810
2,Q-0039,8.554555,0.415931,16.693178
3,Q-0524,0.304160,0.000000,3.311007
4,Q-0741,0.079243,0.000000,2.814545


## 6. Масса на 1 января 2026 года

Для каждого слоя прогнозируем концентрацию в центрах ячеек. Умножаем её на объём воды с учётом пористости и суммируем. Это оценка растворённой массы.

In [7]:
def estimate_mass(model, build_features, raster, layers):
    rows, cols = np.indices(raster.shape)
    x = (cols.ravel() + 0.5) * CELL_SIZE
    y = (rows.ravel() + 0.5) * CELL_SIZE

    mass_mg = 0.0
    for layer in layers.itertuples(index=False):
        depth = (layer.z_top + layer.z_bottom) / 2
        points = pd.DataFrame({"x": x, "y": y, "z": depth, "t": MASS_DATE})
        concentration = predict(model, build_features(points))
        water_m3 = CELL_SIZE**2 * (layer.z_top - layer.z_bottom) * layer.porosity
        # 1 µg/L equals 1 mg/m³.
        mass_mg += concentration.sum() * water_m3
    return float(mass_mg / 1_000_000)

In [8]:
mass_kg = estimate_mass(model, build_features, raster, layers)
print(f"Растворённая масса на {MASS_DATE.date()}: {mass_kg:.2f} кг")

Растворённая масса на 2026-01-01: 142.86 кг


## 7. Порог на водозаборе

Строим ежедневный прогноз для трёх скважин. При смешивании вес каждой скважины равен её расходу. Ищем первую дату, когда итоговая концентрация строго выше 5 мкг/л.

In [9]:
def forecast_intake(model, build_features, wells, pumping, end_date):
    intake = wells.loc[wells["role"].eq("intake"), ["well_id", "x", "y", "z"]]
    days = pd.DataFrame({"t": pd.date_range(MASS_DATE, end_date, freq="D")})
    requests = days.merge(intake, how="cross")
    requests["month"] = month_start(requests["t"])
    rates = pumping[["well_id", "t", "pumping_rate"]].rename(columns={"t": "month"})
    requests = requests.merge(rates, on=["well_id", "month"], how="left", validate="many_to_one")
    if intake.empty or requests["pumping_rate"].isna().any():
        raise ValueError("Missing intake wells or monthly pumping schedule")

    requests["mass_mg_day"] = predict(model, build_features(requests)) * requests["pumping_rate"]
    daily = requests.groupby("t", as_index=False)[["mass_mg_day", "pumping_rate"]].sum()
    daily["prediction"] = daily["mass_mg_day"] / daily["pumping_rate"].where(daily["pumping_rate"] > 0)
    return daily[["t", "pumping_rate", "prediction"]]

In [10]:
intake_forecast = forecast_intake(model, build_features, wells, pumping, queries["t"].max())
arrival = intake_forecast.loc[intake_forecast["prediction"] > THRESHOLD, "t"].min()

if pd.isna(arrival):
    print(f"До {intake_forecast['t'].max().date()} превышения не предсказано.")
else:
    print(f"Первое превышение: {arrival.date()}, через {(arrival - MASS_DATE).days} дней.")
display(intake_forecast.head())

Первое превышение: 2026-01-01, через 0 дней.


,t,pumping_rate,prediction
0,2026-01-01,1214.5,7.956568
1,2026-01-02,1214.5,7.970845
2,2026-01-03,1214.5,7.985119
3,2026-01-04,1214.5,7.999388
4,2026-01-05,1214.5,8.013652


## 8. Сохранение прогноза и модели

В конце заново читаем сохранённую модель и проверяем, что она воспроизводит прогнозы и интервалы.

In [11]:
OUTPUT.mkdir(parents=True, exist_ok=True)

predictions.to_csv(OUTPUT / "predictions.csv", index=False)
intake_forecast.to_csv(OUTPUT / "intake_forecast.csv", index=False)
summary = {
    "mass_kg": mass_kg,
    "arrival_date": None if pd.isna(arrival) else arrival.date().isoformat(),
}
(OUTPUT / "summary.json").write_text(
    json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)

parameter_names = [
    "x0", "y0", "depth_halo", "depth_plume", "log_halo", "log_plume",
    "halo_xy", "halo_z", "length", "width", "plume_z", "speed", "angle",
    "length_growth", "width_growth", "background",
]
model_state = {
    "start_date": START.date().isoformat(),
    "parameters": dict(zip(parameter_names, model.tolist())),
    "interval_factor": interval_factor,
}
(OUTPUT / "model.json").write_text(
    json.dumps(model_state, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)

restored = json.loads((OUTPUT / "model.json").read_text(encoding="utf-8"))
restored_model = np.array([restored["parameters"][name] for name in parameter_names])
np.testing.assert_allclose(
    np.column_stack(predict_interval(restored_model, X_query, restored["interval_factor"])),
    predictions[["prediction", "lower_90", "upper_90"]].to_numpy(),
)
print(f"Файлы сохранены в {OUTPUT}")

Файлы сохранены в D:\WorkSpace\Projects\Task-01-train\submission_plume
